<a href="https://www.kaggle.com/code/airzip/hf-sft-lora-cls-ner-deepseek-distill?scriptVersionId=223844315" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 大模型及系统环境准备

## 系统环境安装

In [ ]:
!pip install -q swanlab peft bitsandbytes

## 基础模型准备

In [2]:
#from modelscope import snapshot_download, AutoTokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
import torch

# 在modelscope上下载Qwen模型到本地目录下
#snapshot_download("qwen/Qwen2-0.5B-Instruct", cache_dir=".", revision="master")

model_id = "/kaggle/input/deepseek-r1/transformers/deepseek-r1-distill-qwen-32b/2"
# Transformers加载模型权重
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", trust_remote_code=True,
    torch_dtype=torch.bfloat16,  # 设置模型的数据类型为torch.bfloat16
    load_in_4bit=True  # 使用4位加载模型
)

base_model.enable_input_require_grads()  # 开启梯度检查点时，要执行该方法

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

量化模型

In [ ]:
"""
from transformers import AutoTokenizer,AutoConfig, AutoModel, BitsAndBytesConfig
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

bnb_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True, #QLoRA 设计的 Double Quantization
            bnb_4bit_quant_type="nf4", #QLoRA 设计的 Normal Float 4 量化数据类型
            llm_int8_threshold=6.0,
            llm_int8_has_fp16_weight=False,
        )

tokenizer = AutoTokenizer.from_pretrained( "Qwen/Qwen2.5-3B-Instruct",trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct",quantization_config=bnb_config,
    low_cpu_mem_usage=True,trust_remote_code=True
)
"""

# 大语言模型微调CLS


数据集：新闻数据

显存要求：10GB左右  

实验过程看：https://swanlab.cn/

## 数据集清洗

In [5]:
# 加载、处理数据集和测试集
import pandas as pd
train_dataset_path = "/kaggle/input/zh-cls-fudan-news/train.jsonl"
test_dataset_path = "/kaggle/input/zh-cls-fudan-news/test.jsonl"
pd.read_json(train_dataset_path, lines=True)

,text,category,output
0,【 文献号 】4-2054\t【原文出处】史学月刊\t【原刊地名】开封\t【原刊期号】199...,"[Electronics, Politics, Economy, Military, Art...",History
1,【 文献号 】3-2104\t【原文出处】《幼儿教育》\t【原刊地名】杭州\t【原刊期号】1...,"[Sports, Communication, Military, Electronics,...",Sports
2,中国环境科学CHINA ENVIRONMENTAL SCIENCE1998年 第18卷 第1...,"[Art, Military, Agriculture, Enviornment, Spac...",Enviornment
3,（兼晚报）我国邮集首次在世界邮展上获金奖新华社北京５月１５日电（记者李玫）我国著名集邮家沈曾...,"[Communication, Art, Politics, Medical, Educat...",Art
4,【 文献号 】7-464\t【原文出处】世界经济与政治\t【原刊地名】京\t【原刊期号】20...,"[Education, Politics]",Politics
...,...,...,...
3995,【 日 期 】19960319\t【 版 号 】10\t【 标 题 】电脑专卖俏京城\...,"[Literature, Military, Philosophy, Art, Sports...",Computer
3996,【 文献号 】3-2769\t【原文出处】光明日报\t【原刊地名】京\t【原刊期号】1997...,"[Law, Computer, Medical, Military, Space, Art,...",History
3997,【 文献号 】2-431\t【原文出处】复旦学报：社科版\t【原刊地名】沪\t【原刊期号】2...,"[Sports, Literature, Education, Economy, Medic...",Economy
3998,【 文献号 】3-2096\t【原文出处】幼儿教育\t【原刊地名】杭州\t【原刊期号】199...,"[Space, Politics, Transport, History, Philosop...",Sports


In [6]:
# 将train.jsonl和test.jsonl进行处理，转换成new_train.jsonl和new_test.jsonl

import json
import pandas as pd
import os

def dataset_jsonl_transfer(origin_path, new_path):
    """
    将原始数据集转换为大模型微调所需数据格式的新数据集
    """
    messages = []

    # 读取旧的JSONL文件
    with open(origin_path, "r") as file:
        for line in file:
            # 解析每一行的json数据
            data = json.loads(line)
            context = data["text"]
            catagory = data["category"]
            label = data["output"]
            message = {
                "instruction": "你是一个文本分类领域的专家，你会接收到一段文本和几个潜在的分类选项，请输出文本内容的正确类型",
                "input": f"文本:{context},类型选型:{catagory}",
                "output": label,
            }
            messages.append(message)

    # 保存重构后的JSONL文件
    with open(new_path, "w", encoding="utf-8") as file:
        for message in messages:
            file.write(json.dumps(message, ensure_ascii=False) + "\n")


In [7]:
train_jsonl_new_path = "/tmp/new_train.jsonl"
test_jsonl_new_path = "/tmp/new_test.jsonl"

if not os.path.exists(train_jsonl_new_path):
    dataset_jsonl_transfer(train_dataset_path, train_jsonl_new_path)
if not os.path.exists(test_jsonl_new_path):
    dataset_jsonl_transfer(test_dataset_path, test_jsonl_new_path)

train_df = pd.read_json(train_jsonl_new_path, lines=True)[:1000]  # 取前1000条做训练（可选）
test_df = pd.read_json(test_jsonl_new_path, lines=True)[:10]  # 取前10条做主观评测
train_df[:3]

,instruction,input,output
0,你是一个文本分类领域的专家，你会接收到一段文本和几个潜在的分类选项，请输出文本内容的正确类型,文本:【 文献号 】4-2054\t【原文出处】史学月刊\t【原刊地名】开封\t【原刊期号】...,History
1,你是一个文本分类领域的专家，你会接收到一段文本和几个潜在的分类选项，请输出文本内容的正确类型,文本:【 文献号 】3-2104\t【原文出处】《幼儿教育》\t【原刊地名】杭州\t【原刊期...,Sports
2,你是一个文本分类领域的专家，你会接收到一段文本和几个潜在的分类选项，请输出文本内容的正确类型,文本:中国环境科学CHINA ENVIRONMENTAL SCIENCE1998年 第18卷...,Enviornment


## 训练语料预处理

In [8]:
def process_func(example):
    """
    将数据集进行预处理
    """
    MAX_LENGTH = 384
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer(
        # 加入Qwen特殊符防提示词注入
        f"<|im_start|>system\n你是一个文本分类领域的专家，你会接收到一段文本和几个潜在的分类选项，请输出文本内容的正确类型<|im_end|>\n<|im_start|>user\n{example['input']}<|im_end|>\n<|im_start|>assistant\n",
        add_special_tokens=False,
    )
    response = tokenizer(f"{example['output']}", add_special_tokens=False)
    input_ids = (
        instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
    )
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]
    labels = (
        [-100] * len(instruction["input_ids"])
        + response["input_ids"]
        + [tokenizer.pad_token_id]
    )
    if len(input_ids) > MAX_LENGTH:  # 做截断
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
        
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [9]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
train_dataset = train_ds.map(process_func, remove_columns=train_ds.column_names, num_proc=4)
train_dataset

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (18772 > 16384). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (19593 > 16384). Running this sequence through the model will result in indexing errors


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})

## 设置低秩矩阵参数

In [10]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    inference_mode=False,  # 训练模式
    r=8,  # Lora 秩
    lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1,  # Dropout 比例
)

model_cls = get_peft_model(base_model, config)
#model_cls

## 配置超参及训练

In [11]:
args = TrainingArguments(
    output_dir="./output/SFT-CLS",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=6,
    logging_steps=5,
    num_train_epochs=2,
    save_steps=100,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none"
)

In [12]:
from swanlab.integration.huggingface import SwanLabCallback
import swanlab

swanlab_callback = SwanLabCallback(
    project="DeepSeek-R1",
    experiment_name="SFT-CLS",
    description="320亿模型在新闻数据集上微调",
    config={
        "model": "Deepseek-R1-distill-qwen-32B",
        "dataset": "zh_cls_fudan-news",
    },
)

<ipython-input-12-44fea499623c>:1: DeprecationWarning: The module 'huggingface' is deprecated and will be removed in future versions. Please update your imports to use 'transformers' instead.
  from swanlab.integration.huggingface import SwanLabCallback


In [ ]:
trainer = Trainer(
    model=model_cls,
    args=args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    #callbacks=[swanlab_callback],
)

trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
5,1986.776600


In [ ]:
# 如果训练中断了，还可以从上次中断保存的位置继续开始训练
if train_with_checkpoint:
    checkpoint = [file for file in os.listdir(output_dir) if 'checkpoint' in file][-1]
    last_checkpoint = f'{output_dir}/{checkpoint}'
    print(last_checkpoint)
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    trainer.train()

In [ ]:
## 保存 LoRa 权重和分词
trainer.model.save_pretrained('./model-lora')
tokenizer.save_pretrained('./model-lora')

## 模型合并推理模式

In [ ]:
from peft import PeftModel

lora_model_cls = PeftModel.from_pretrained(
    base_model, model_id='/kaggle/working/model-lora',
    #config=config, 
    is_trainable=False
)

In [ ]:
new_model_cls = lora_model_cls.merge_and_unload()
new_model_cls.eval()

## 模型预测效果评估

In [ ]:
# ====== 训练结束后的预测 ===== #

def predict(messages, model, tokenizer):
    device = "cuda"
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    
    generated_ids = model.generate(model_inputs["input_ids"], max_new_tokens=512)
    generated_ids = [
        output_ids[len(input_ids) :]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(response)

    return response

In [ ]:
test_df[:3]

In [ ]:
valid_df = train_df.sample(frac=0.01)
valid_df

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
import swanlab
test_text_list = []
for index, row in valid_df.iterrows():
    instruction = row["instruction"]
    input_value = row["input"]

    messages = [
        {"role": "system", "content": f"{instruction}"},
        {"role": "user", "content": f"{input_value}"},
    ]

    response = predict(messages, new_model_cls, tokenizer)
    messages.append({"role": "assistant", "content": f"{response}"})
    result_text = f"{messages[1]}\n\n{messages[2]}"
    test_text_list.append(swanlab.Text(result_text, caption=response))

In [ ]:
import swanlab
swanlab.init() 
swanlab.log({"Prediction": test_text_list})
swanlab.finish()

# 大语言模型微调NER

数据集：新闻数据

显存要求：10GB左右  

实验过程看：https://swanlab.cn/

## 准备数据

In [ ]:
!huggingface-cli download --repo-type dataset qgyd2021/chinese_ner_sft --local-dir ./qgyd2021
from IPython.display import clear_output
clear_output()

In [ ]:
import pandas as pd
train_dataset_path = "/kaggle/working/data/ccfbdci.jsonl"
pd.read_json('/kaggle/working/data/ccfbdci.jsonl', lines=True)

In [ ]:
# 2.将train.jsonl和test.jsonl进行处理，转换成new_train.jsonl和new_test.jsonl

import json
import pandas as pd
import os

def dataset_jsonl_transfer(origin_path, new_path):
    """
    将原始数据集转换为大模型微调所需数据格式的新数据集
    """
    messages = []

    # 读取旧的JSONL文件
    with open(origin_path, "r") as file:
        for line in file:
            # 解析每一行的json数据
            data = json.loads(line)
            input_text = data["text"]
            entities = data["entities"]
            match_names = ["地点", "人名", "地理实体", "组织"]
            
            entity_sentence = ""
            for entity in entities:
                entity_json = dict(entity)
                entity_text = entity_json["entity_text"]
                entity_names = entity_json["entity_names"]
                for name in entity_names:
                    if name in match_names:
                        entity_label = name
                        break
                
                entity_sentence += f"""{{"entity_text": "{entity_text}", "entity_label": "{entity_label}"}}"""
            
            if entity_sentence == "":
                entity_sentence = "没有找到任何实体"
            
            message = {
                "instruction": """你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {"entity_text": "南京", "entity_label": "地理实体"} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出"没有找到任何实体". """,
                "input": f"文本:{input_text}",
                "output": entity_sentence,
            }
            
            messages.append(message)

    # 保存重构后的JSONL文件
    with open(new_path, "w", encoding="utf-8") as file:
        for message in messages:
            file.write(json.dumps(message, ensure_ascii=False) + "\n")


# 加载、处理数据集和测试集
train_jsonl_new_path = "ccf_train.jsonl"

if not os.path.exists(train_jsonl_new_path):
    dataset_jsonl_transfer(train_dataset_path, train_jsonl_new_path)

total_df = pd.read_json(train_jsonl_new_path, lines=True)
train_df = total_df[int(len(total_df) * 0.1):]  # 取90%的数据做训练集
test_df = total_df[:int(len(total_df) * 0.1)].sample(n=20)  # 随机取10%的数据中的20条做测试集
train_df

## 语料分词预处理

In [ ]:
def process_func(example):
    """
    将数据集进行预处理, 处理成模型可以接受的格式
    """

    MAX_LENGTH = 384 
    input_ids, attention_mask, labels = [], [], []
    system_prompt = """你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {"entity_text": "南京", "entity_label": "地理实体"} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出"没有找到任何实体"."""
    
    instruction = tokenizer(
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{example['input']}<|im_end|>\n<|im_start|>assistant\n",
        add_special_tokens=False,
    )
    response = tokenizer(f"{example['output']}", add_special_tokens=False)
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
    attention_mask = (
        instruction["attention_mask"] + response["attention_mask"] + [1]
    )
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.pad_token_id]
    if len(input_ids) > MAX_LENGTH:  # 做一个截断
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}   

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
train_dataset = train_ds.map(process_func, remove_columns=train_ds.column_names, num_proc=4)

## 设置低秩矩阵参数

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    inference_mode=False,  # 训练模式
    r=8,  # Lora 秩
    lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1,  # Dropout 比例
)

lora_model_ner = get_peft_model(base_model, config)

## 超参配置可视化及训练

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir="./output/Qwen2-NER",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    logging_steps=10,
    num_train_epochs=2,
    save_steps=100,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none",
)

In [ ]:
torch.cuda.empty_cache()
base_model.enable_input_require_grads()

In [ ]:
from swanlab.integration.huggingface import SwanLabCallback
import swanlab

swanlab_callback = SwanLabCallback(
    project="Qwen",
    experiment_name="Qwen2-0.5B-Instruct",
    description="用通义千问Qwen2-7B-Instruct量化模型在NER数据集上微调，实现关键实体识别任务。",
    config={
        "model": "Qwen2-0.5B-Instruct",
        #"model_dir": model_dir,
        "dataset": "qgyd2021/chinese_ner_sft",
    },
)

In [ ]:
trainer = Trainer(
    model=lora_model_ner,
    args=args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    callbacks=[swanlab_callback],
)

trainer.train()

In [ ]:
## 保存 LoRa 权重和分词
trainer.model.save_pretrained('./model-lora-ner')
tokenizer.save_pretrained('./model-lora-ner')

## 融合模型及持久化

In [ ]:
from peft import PeftModel

lora_model_ner = PeftModel.from_pretrained(
    base_model, model_id='./model-lora-ner',
    #config=config, 
    is_trainable=False
)

In [ ]:
new_model_ner = lora_model_ner.merge_and_unload()
new_model_ner.eval()

## 微调新大模型评测

In [ ]:
# ====== 训练结束后的预测 ===== #

def predict(messages, model, tokenizer):
    device = "cuda"
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    generated_ids = model.generate(model_inputs.input_ids, max_new_tokens=512)
    generated_ids = [
        output_ids[len(input_ids) :]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(response)

    return response

In [ ]:

test_text_list = []
for index, row in test_df.iterrows():
    instruction = row["instruction"]
    input_value = row["input"]

    messages = [
        {"role": "system", "content": f"{instruction}"},
        {"role": "user", "content": f"{input_value}"},
    ]

    response = predict(messages, new_model_ner, tokenizer)
    messages.append({"role": "assistant", "content": f"{response}"})
    result_text = f"{messages[0]}\n\n{messages[1]}\n\n{messages[2]}"
    test_text_list.append(swanlab.Text(result_text, caption=response))

In [ ]:
swanlab.log({"Prediction": test_text_list})
swanlab.finish()